## 05 Purchase Intent Prediction

Predict the likelihood that a customer will purchase a specific product using customer behavior, product attributes, enriched marketing features, and customer segment information.

Output:

- trained XGBoost purchase intent model
- feature columns used for inference
- feature importance table
- customer-product purchase probability predictions


In [1]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

#### Load Data


In [2]:
PROCESSED_DIR = "../data/processed/hm"
MODEL_DIR = "../saved_models/purchase_model"

os.makedirs(MODEL_DIR, exist_ok=True)

transactions = pd.read_csv(os.path.join(PROCESSED_DIR, "transactions.csv"))
articles = pd.read_csv(os.path.join(PROCESSED_DIR, "articles_enriched.csv"))
customers = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_segments.csv"))

print(transactions.shape)
print(articles.shape)
print(customers.shape)

(500000, 5)
(105542, 37)
(317897, 27)


In [3]:
articles.head()

,article_id,product_code,product_name,product_type_no,product_type,product_group,graphical_appearance_no,graphical_appearance,colour_group_code,color_group,...,product_purchase_count,unique_customer_count,avg_selling_price,style,occasion,material_hint,target_audience,selling_points,marketing_keywords,copy_angle
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,175.0,172.0,0.008139,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'Black']",everyday and daily_wear focused
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,116.0,116.0,0.008196,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'White']",everyday and daily_wear focused
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,2.0,2.0,0.004559,everyday,daily_wear,unknown,Ladieswear,"['Easy to style Vest top', 'Versatile for dail...","['everyday', 'daily_wear', 'Off White']",everyday and daily_wear focused
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,19.0,19.0,0.020756,elegant,work_or_outing,unknown,Ladieswear,"['Easy to style Bra', 'Versatile for work_or_o...","['elegant', 'work_or_outing', 'Black']",elegant and work_or_outing focused
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,11.0,11.0,0.016932,elegant,work_or_outing,unknown,Ladieswear,"['Easy to style Bra', 'Versatile for work_or_o...","['elegant', 'work_or_outing', 'White']",elegant and work_or_outing focused


In [4]:
transactions.head()

,transaction_date,customer_id,article_id,price,sales_channel_id
0,2019-09-13,215895f90002eb3d1a04bd603513c8e85e6002ef08f136...,786586001,0.022017,1
1,2019-02-23,7b183268e3a4623b80d5325ec4a20a0af0edff7bcb1748...,658911001,0.028797,2
2,2019-07-17,2eb7412239a90c0570cd3d1bf0492856ae5b59058b1ea6...,759326005,0.050831,2
3,2019-05-16,74f162e5a170fd57207aa2a7d5c58479ee9de903b2a277...,737137004,0.027102,1
4,2019-08-10,aab9306ee28c4db494003955f80355e540b01480ab35cf...,785931001,0.050831,2


In [5]:
customers.head()

,customer_id,total_transactions,unique_products,total_spend,avg_price,max_price,first_purchase_date,last_purchase_date,days_since_last_purchase,customer_lifetime_days,...,fashion_news_binary,is_active,club_member_status,fashion_news_frequency,age,age_group,cluster,customer_segment,pca_1,pca_2
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,1,1,0.012695,0.012695,0.012695,2019-07-25,2019-07-25,426,1,...,0.0,0.0,ACTIVE,NONE,49.0,Adult,1,Inactive Budget Shoppers,-2.093230,-0.994263
1,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1,1,0.044475,0.044475,0.044475,2019-10-01,2019-10-01,358,1,...,1.0,1.0,ACTIVE,Regularly,52.0,Mature,0,High-Value One-Time Buyers,0.180387,2.007445
2,0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...,1,1,0.022017,0.022017,0.022017,2019-10-22,2019-10-22,337,1,...,0.0,0.0,ACTIVE,NONE,20.0,Gen Z,1,Inactive Budget Shoppers,-1.342126,0.156293
3,00007d2de826758b65a93dd24ce629ed66842531df6699...,2,2,0.032847,0.016424,0.017610,2018-09-20,2020-04-11,165,570,...,1.0,1.0,ACTIVE,Regularly,32.0,Young Adult,3,Regular Shoppers,1.374438,-2.806089
4,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,2,2,0.040932,0.020466,0.025407,2020-06-21,2020-08-17,37,58,...,1.0,1.0,ACTIVE,Regularly,56.0,Mature,3,Regular Shoppers,0.624498,-1.714984


#### Create Positive Samples


In [6]:
positive_samples = transactions[["customer_id", "article_id"]].copy()
positive_samples["purchased"] = 1

positive_samples = positive_samples.drop_duplicates()

positive_samples.head()

,customer_id,article_id,purchased
0,215895f90002eb3d1a04bd603513c8e85e6002ef08f136...,786586001,1
1,7b183268e3a4623b80d5325ec4a20a0af0edff7bcb1748...,658911001,1
2,2eb7412239a90c0570cd3d1bf0492856ae5b59058b1ea6...,759326005,1
3,74f162e5a170fd57207aa2a7d5c58479ee9de903b2a277...,737137004,1
4,aab9306ee28c4db494003955f80355e540b01480ab35cf...,785931001,1


#### Create Negative Samples


In [17]:
# No Sampling: Random sampling + filtering
np.random.seed(42)

# Use full dataset
positive_samples_full = positive_samples.copy()

all_articles = articles["article_id"].dropna().unique()

# Step 1: Create negative samples by random pairing
negative_samples = positive_samples_full.copy()

negative_samples["article_id"] = np.random.choice(
    all_articles,
    size=len(positive_samples_full),
    replace=True
)

negative_samples["purchased"] = 0

# Step 2: Build lookup for real purchases
purchased_pairs = set(zip(
    positive_samples_full["customer_id"],
    positive_samples_full["article_id"]
))

# Step 3: Remove accidental positives
negative_samples = negative_samples[
    ~negative_samples[["customer_id", "article_id"]]
    .apply(tuple, axis=1)
    .isin(purchased_pairs)
].copy()

# Step 4: Combine
modeling_pairs = pd.concat(
    [positive_samples_full, negative_samples],
    ignore_index=True
)

# Step 5: Remove duplicates
modeling_pairs = modeling_pairs.drop_duplicates(
    subset=["customer_id", "article_id", "purchased"]
).copy()

print(modeling_pairs["purchased"].value_counts())
print(modeling_pairs.shape)

purchased
1    498294
0    498273
Name: count, dtype: int64
(996567, 3)


In [18]:
# Check overlap
positive_pairs = set(zip(
    modeling_pairs[modeling_pairs["purchased"] == 1]["customer_id"],
    modeling_pairs[modeling_pairs["purchased"] == 1]["article_id"]
))

negative_pairs = set(zip(
    modeling_pairs[modeling_pairs["purchased"] == 0]["customer_id"],
    modeling_pairs[modeling_pairs["purchased"] == 0]["article_id"]
))

overlap = positive_pairs.intersection(negative_pairs)

print("Overlap count:", len(overlap))

Overlap count: 0


In [19]:
# Check duplicates
duplicates = modeling_pairs.duplicated(
    subset=["customer_id", "article_id", "purchased"]
).sum()

print("Duplicate rows:", duplicates)

Duplicate rows: 0


In [20]:
# Check same (customer, article) appearing twice with different labels
pair_counts = modeling_pairs.groupby(
    ["customer_id", "article_id"]
)["purchased"].nunique()

conflicts = (pair_counts > 1).sum()

print("Conflicting labels:", conflicts)

Conflicting labels: 0


#### Build Modeling Dataset

customer + product + features + purchased label


In [21]:
# Defines which customer and product columns will be used as model features
customer_feature_cols = [
    "customer_id",
    "total_transactions",
    "unique_products",
    "total_spend",
    "avg_price",
    "max_price",
    "days_since_last_purchase",
    "customer_lifetime_days",
    "purchase_frequency",
    "low_price_purchase_ratio",
    "high_price_purchase_ratio",
    "fashion_news_binary",
    "is_active",
    "age",
    "age_group",
    "customer_segment"
]

product_feature_cols = [
    "article_id",
    "product_type",
    "product_group",
    "color_group",
    "index_group",
    "garment_group",
    "avg_selling_price",
    "style",
    "occasion",
    "material_hint",
    "target_audience",
    "copy_angle"
]

In [22]:
# Adds customer features and product features to each customer-product pair
modeling_df = modeling_pairs.merge(
    customers[customer_feature_cols],
    on="customer_id",
    how="left"
)

modeling_df = modeling_df.merge(
    articles[product_feature_cols],
    on="article_id",
    how="left"
)

modeling_df.head()

,customer_id,article_id,purchased,total_transactions,unique_products,total_spend,avg_price,max_price,days_since_last_purchase,customer_lifetime_days,...,product_group,color_group,index_group,garment_group,avg_selling_price,style,occasion,material_hint,target_audience,copy_angle
0,215895f90002eb3d1a04bd603513c8e85e6002ef08f136...,786586001,1,1,1,0.022017,0.022017,0.022017,376,1,...,Garment Lower body,Black,Baby/Children,Jersey Fancy,0.019414,everyday,daily_wear,unknown,Baby/Children,everyday and daily_wear focused
1,7b183268e3a4623b80d5325ec4a20a0af0edff7bcb1748...,658911001,1,5,5,0.128051,0.025610,0.030492,143,436,...,Nightwear,Dark Blue,Ladieswear,"Under-, Nightwear",0.028797,everyday,daily_wear,unknown,Ladieswear,everyday and daily_wear focused
2,2eb7412239a90c0570cd3d1bf0492856ae5b59058b1ea6...,759326005,1,2,2,0.091492,0.045746,0.050831,361,74,...,Swimwear,Dark Orange,Ladieswear,Swimwear,0.045862,everyday,vacation,unknown,Ladieswear,everyday and vacation focused
3,74f162e5a170fd57207aa2a7d5c58479ee9de903b2a277...,737137004,1,2,2,0.065220,0.032610,0.038119,361,136,...,Garment Upper body,Dark Green,Ladieswear,Blouses,0.022032,elegant,work_or_outing,unknown,Ladieswear,elegant and work_or_outing focused
4,aab9306ee28c4db494003955f80355e540b01480ab35cf...,785931001,1,1,1,0.050831,0.050831,0.050831,410,1,...,Garment Lower body,White,Ladieswear,Trousers,0.046932,everyday,daily_wear,unknown,Ladieswear,everyday and daily_wear focused


#### Interaction Features


In [23]:
# Compares product price with the customer’s average purchase price
modeling_df["price_vs_customer_avg"] = (
    modeling_df["avg_selling_price"] - modeling_df["avg_price"]
)

# Creates simple binary indicators
modeling_df["is_lower_than_customer_avg"] = (
    modeling_df["avg_selling_price"] < modeling_df["avg_price"]
).astype(int)

modeling_df["is_higher_than_customer_avg"] = (
    modeling_df["avg_selling_price"] > modeling_df["avg_price"]
).astype(int)

modeling_df.head()

,customer_id,article_id,purchased,total_transactions,unique_products,total_spend,avg_price,max_price,days_since_last_purchase,customer_lifetime_days,...,garment_group,avg_selling_price,style,occasion,material_hint,target_audience,copy_angle,price_vs_customer_avg,is_lower_than_customer_avg,is_higher_than_customer_avg
0,215895f90002eb3d1a04bd603513c8e85e6002ef08f136...,786586001,1,1,1,0.022017,0.022017,0.022017,376,1,...,Jersey Fancy,0.019414,everyday,daily_wear,unknown,Baby/Children,everyday and daily_wear focused,-0.002603,1,0
1,7b183268e3a4623b80d5325ec4a20a0af0edff7bcb1748...,658911001,1,5,5,0.128051,0.025610,0.030492,143,436,...,"Under-, Nightwear",0.028797,everyday,daily_wear,unknown,Ladieswear,everyday and daily_wear focused,0.003186,0,1
2,2eb7412239a90c0570cd3d1bf0492856ae5b59058b1ea6...,759326005,1,2,2,0.091492,0.045746,0.050831,361,74,...,Swimwear,0.045862,everyday,vacation,unknown,Ladieswear,everyday and vacation focused,0.000116,0,1
3,74f162e5a170fd57207aa2a7d5c58479ee9de903b2a277...,737137004,1,2,2,0.065220,0.032610,0.038119,361,136,...,Blouses,0.022032,elegant,work_or_outing,unknown,Ladieswear,elegant and work_or_outing focused,-0.010578,1,0
4,aab9306ee28c4db494003955f80355e540b01480ab35cf...,785931001,1,1,1,0.050831,0.050831,0.050831,410,1,...,Trousers,0.046932,everyday,daily_wear,unknown,Ladieswear,everyday and daily_wear focused,-0.003898,1,0


#### Prepare Features


In [24]:
# Defines the target and removes ID columns from model input
target_col = "purchased"

drop_cols = [
    "customer_id",
    "article_id",
    target_col
]

X = modeling_df.drop(columns=drop_cols)
y = modeling_df[target_col]

X.head()

,total_transactions,unique_products,total_spend,avg_price,max_price,days_since_last_purchase,customer_lifetime_days,purchase_frequency,low_price_purchase_ratio,high_price_purchase_ratio,...,garment_group,avg_selling_price,style,occasion,material_hint,target_audience,copy_angle,price_vs_customer_avg,is_lower_than_customer_avg,is_higher_than_customer_avg
0,1,1,0.022017,0.022017,0.022017,376,1,1.000000,0.0,0.0,...,Jersey Fancy,0.019414,everyday,daily_wear,unknown,Baby/Children,everyday and daily_wear focused,-0.002603,1,0
1,5,5,0.128051,0.025610,0.030492,143,436,0.011468,0.0,0.0,...,"Under-, Nightwear",0.028797,everyday,daily_wear,unknown,Ladieswear,everyday and daily_wear focused,0.003186,0,1
2,2,2,0.091492,0.045746,0.050831,361,74,0.027027,0.0,1.0,...,Swimwear,0.045862,everyday,vacation,unknown,Ladieswear,everyday and vacation focused,0.000116,0,1
3,2,2,0.065220,0.032610,0.038119,361,136,0.014706,0.0,0.5,...,Blouses,0.022032,elegant,work_or_outing,unknown,Ladieswear,elegant and work_or_outing focused,-0.010578,1,0
4,1,1,0.050831,0.050831,0.050831,410,1,1.000000,0.0,1.0,...,Trousers,0.046932,everyday,daily_wear,unknown,Ladieswear,everyday and daily_wear focused,-0.003898,1,0


In [25]:
# Separates categorical and numeric columns
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

Categorical columns: ['age_group', 'customer_segment', 'product_type', 'product_group', 'color_group', 'index_group', 'garment_group', 'style', 'occasion', 'material_hint', 'target_audience', 'copy_angle']
Numeric columns: ['total_transactions', 'unique_products', 'total_spend', 'avg_price', 'max_price', 'days_since_last_purchase', 'customer_lifetime_days', 'purchase_frequency', 'low_price_purchase_ratio', 'high_price_purchase_ratio', 'fashion_news_binary', 'is_active', 'age', 'avg_selling_price', 'price_vs_customer_avg', 'is_lower_than_customer_avg', 'is_higher_than_customer_avg']


In [26]:
# Converts categorical features into numeric dummy variables
X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    dummy_na=True
)

X_encoded = X_encoded.fillna(0)

X_encoded.head()

,total_transactions,unique_products,total_spend,avg_price,max_price,days_since_last_purchase,customer_lifetime_days,purchase_frequency,low_price_purchase_ratio,high_price_purchase_ratio,...,target_audience_Sport,target_audience_nan,copy_angle_casual and daily_wear focused,copy_angle_elegant and daily_wear focused,copy_angle_elegant and work_or_outing focused,copy_angle_everyday and daily_wear focused,copy_angle_everyday and vacation focused,copy_angle_sporty and workout focused,copy_angle_streetwear and daily_wear focused,copy_angle_nan
0,1,1,0.022017,0.022017,0.022017,376,1,1.000000,0.0,0.0,...,False,False,False,False,False,True,False,False,False,False
1,5,5,0.128051,0.025610,0.030492,143,436,0.011468,0.0,0.0,...,False,False,False,False,False,True,False,False,False,False
2,2,2,0.091492,0.045746,0.050831,361,74,0.027027,0.0,1.0,...,False,False,False,False,False,False,True,False,False,False
3,2,2,0.065220,0.032610,0.038119,361,136,0.014706,0.0,0.5,...,False,False,False,False,True,False,False,False,False,False
4,1,1,0.050831,0.050831,0.050831,410,1,1.000000,0.0,1.0,...,False,False,False,False,False,True,False,False,False,False


#### Train-Test Split


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y # Keeps the purchase / non-purchase ratio similar in both train and test sets
)

print(X_train.shape)
print(X_test.shape)

(797253, 285)
(199314, 285)


#### Train XGBoost Model

customer behavior + segment + product attributes → probability of purchase


In [28]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


#### Evaluate Model


In [29]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.91      0.77      0.84     99655
           1       0.80      0.93      0.86     99659

    accuracy                           0.85    199314
   macro avg       0.86      0.85      0.85    199314
weighted avg       0.86      0.85      0.85    199314

ROC-AUC: 0.9256010154830673


In [30]:
confusion_matrix(y_test, y_pred)

array([[77149, 22506],
       [ 7187, 92472]])

#### Feature Importance


In [31]:
feature_importance = pd.DataFrame({
    "feature": X_encoded.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(20)

,feature,importance
13,avg_selling_price,0.093910
230,index_group_Baby/Children,0.087839
274,target_audience_Menswear,0.058979
14,price_vs_customer_avg,0.057739
271,target_audience_Baby/Children,0.049778
1,unique_products,0.044052
233,index_group_Menswear,0.033191
17,age_group_Adult,0.028208
159,product_group_Accessories,0.025356
236,garment_group_Accessories,0.023896


#### Save Model Outputs


In [32]:
model_path = os.path.join(MODEL_DIR, "xgboost_purchase_intent_model.pkl")
feature_columns_path = os.path.join(MODEL_DIR, "purchase_model_features.pkl")
importance_path = os.path.join(PROCESSED_DIR, "purchase_model_feature_importance.csv")

joblib.dump(model, model_path)
joblib.dump(X_encoded.columns.tolist(), feature_columns_path)

feature_importance.to_csv(importance_path, index=False)

print(model_path)
print(feature_columns_path)
print(importance_path)

../saved_models/purchase_model/xgboost_purchase_intent_model.pkl
../saved_models/purchase_model/purchase_model_features.pkl
../data/processed/hm/purchase_model_feature_importance.csv


#### Generate Sample Predictions


In [33]:
sample_predictions = modeling_df[["customer_id", "article_id", "purchased"]].copy()
sample_predictions["purchase_probability"] = model.predict_proba(X_encoded)[:, 1]

sample_predictions = sample_predictions.sort_values(
    "purchase_probability",
    ascending=False
)

sample_predictions.head()

,customer_id,article_id,purchased,purchase_probability
120210,fa42cd450ec2cfe6883cefa1347e60e46f49eee236975b...,762656001,1,0.996473
315082,00969d7914b80829cf7263f8a0848bc97c7ccc8687ce29...,762656001,1,0.995825
235425,9c7ed970d91a76017f7b6c70740606147c59930dbf85a2...,762656002,1,0.995405
71101,6c3d2841f7058cfc4a8cb33f662669c0109a58ae03f807...,762656001,1,0.994955
257345,3a9e46ac94fc6c292e28016e1b6ad0279f0523a09afbc1...,762656002,1,0.994575


In [34]:
prediction_path = os.path.join(PROCESSED_DIR, "purchase_intent_predictions.csv")

sample_predictions.to_csv(prediction_path, index=False)

print(prediction_path)

../data/processed/hm/purchase_intent_predictions.csv


#### Output Check


In [35]:
pd.read_csv(prediction_path).head()

,customer_id,article_id,purchased,purchase_probability
0,fa42cd450ec2cfe6883cefa1347e60e46f49eee236975b...,762656001,1,0.996473
1,00969d7914b80829cf7263f8a0848bc97c7ccc8687ce29...,762656001,1,0.995825
2,9c7ed970d91a76017f7b6c70740606147c59930dbf85a2...,762656002,1,0.995405
3,6c3d2841f7058cfc4a8cb33f662669c0109a58ae03f807...,762656001,1,0.994955
4,3a9e46ac94fc6c292e28016e1b6ad0279f0523a09afbc1...,762656002,1,0.994575
